In [1]:
import functools
import jax
import os
from datetime import datetime
from jax import numpy as jp
import matplotlib.pyplot as plt

from IPython.display import HTML, clear_output

try:
    import brax
except ImportError:
    !pip install git+https://github.com/google/brax.git@main
    clear_output()
    import brax

import flax

import flax.serialization
import pickle

from brax import envs
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import train as sac
from brax.io import html, mjcf, model
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
if 'COLAB_TPU_ADDR' in os.environ:
    from jax.tools import colab_tpu
    colab_tpu.setup_tpu()





In [2]:
#@title Check if MuJoCo installation was successful

# from google.colab import files

import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags


Sat Apr  5 14:45:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:17:00.0 Off |                  Off |
| 30%   35C    P0             52W /  450W |    5148MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import time
import itertools
import numpy as np
from typing import Callable, NamedTuple, Optional, Union, List

print('Installing mediapy:')
# ! command -v ffmpeg >/dev/null || (dnf update && dnf install -y ffmpeg)
# ! pip install -q mediapy
# import mediapy as media
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True, linewidth=100)

Installing mediapy:


In [4]:
#@title Import MuJoCo, MJX, and Brax
from datetime import datetime
from etils import epath
import functools
from IPython.display import HTML
from typing import Any, Dict, Sequence, Tuple, Union
import os
from ml_collections import config_dict


import jax
from jax import numpy as jp
import numpy as np
from flax.training import orbax_utils
from flax import struct
from matplotlib import pyplot as plt
import mediapy as media
from orbax import checkpoint as ocp

import mujoco
from mujoco import mjx

from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import html, mjcf, model


# 导入RL模型

In [5]:
#@title Load Model and Define Inference Function
from brax.training.acme import running_statistics
from brax.training.agents.sac import networks as sac_networks
env_name = 'walker2d'
backend = 'positional'
# create an env with auto-reset
env = envs.create(env_name=env_name, backend=backend)
# normalize_fn = lambda x, y: x

model_path = '/u20/li3658/rl_training/models/test4'
params = model.load_params(model_path)
# sac_network = sac_networks.make_sac_networks(
#         env.observation_size, env.action_size, normalize_fn
#     )
sac_network = sac_networks.make_sac_networks(
        action_size=env.action_size, observation_size=env.observation_size, preprocess_observations_fn=running_statistics.normalize
    )
inference = sac_networks.make_inference_fn(sac_network)
jit_inference_fn = inference(params)

jit_inference_fn = jax.jit(jit_inference_fn)

# 定义dataset

In [6]:
from torch.utils.data import Dataset, DataLoader

class WalkerDataset(Dataset):
    def __init__(self, data):
        """
        data: List of tuples (left_leg_data, right_leg_data)
        """
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        left_leg, right_leg = self.data[idx]
        # 解除 `read-only` 并转换为 PyTorch Tensor
        left_leg = torch.tensor(np.array(left_leg, dtype=np.float32).copy())
        right_leg = torch.tensor(np.array(right_leg, dtype=np.float32).copy())
        return left_leg, right_leg



# 加载dataset

In [7]:
import torch.serialization
import numpy as np


torch.serialization.add_safe_globals(["jax._src.array._reconstruct_array"])

loaded_data = torch.load("/u20/li3658/rl_training/dataset/walker2d_dataset.pth", weights_only=False)
# 复用已有的 WalkerDataset
dataset = WalkerDataset(loaded_data)
device = torch.device("cuda")
# dataset.to(device)
# 重新创建 DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=False)


print("✅ WalkerDataset 重新创建成功！")


✅ WalkerDataset 重新创建成功！


# 定义加强版非线形网络

In [ ]:
import torch.nn.functional as F

class EnhancedWalkerGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.2):
        super(EnhancedWalkerGNN, self).__init__()
        # 更深更宽的网络
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim*2)
        self.bn2 = nn.BatchNorm1d(hidden_dim*2)
        self.fc3 = nn.Linear(hidden_dim*2, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)
        
        # 添加残差连接
        self.shortcut = nn.Linear(input_dim, hidden_dim)
        
        # 权重初始化
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
    def forward(self, x):
        # 主路径
        main = F.leaky_relu(self.bn1(self.fc1(x)), negative_slope=0.1)
        main = self.dropout(main)
        main = F.leaky_relu(self.bn2(self.fc2(main)), negative_slope=0.1)
        main = self.dropout(main)
        main = F.leaky_relu(self.bn3(self.fc3(main)), negative_slope=0.1)
        
        # 残差连接
        shortcut = F.leaky_relu(self.shortcut(x), negative_slope=0.1)
        
        # 合并
        combined = main + shortcut
        output = self.fc4(combined)
        
        return output

# 加载训练好的增强版非线形网络

In [1]:

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gnn_model = EnhancedWalkerGNN(input_dim=3, hidden_dim=32, output_dim=3)
# **加载已保存的权重**
# device  = torch.device("cpu")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "/u20/li3658/rl_training/walker_training_results"
checkpoint = torch.load(os.path.join(save_path, 'best_model.pth'))
gnn_model.load_state_dict(checkpoint['model_state_dict'])
gnn_model.eval()


NameError: name 'EnhancedWalkerGNN' is not defined

# 通过RL获取左边腿的action，然后通过增强版非线形模型获取右边腿的action

In [ ]:
#@title Visualizing a trajectory of the learned inference function
import jax
import jax.numpy as jnp
# create an env with auto-reset

env_name = 'walker2d'
backend = 'positional'
env = envs.create(env_name=env_name, backend=backend)

jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
# jit_inference_fn = jax.jit(inference_fn)
data = []
rollout = []
rng = jax.random.PRNGKey(seed=1)
gnn_model.to(device)
gnn_model.eval()
state = jit_env_reset(rng=rng)
for _ in range(1000):
    rollout.append(state.pipeline_state)
    act_rng, rng = jax.random.split(rng)
    act, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_env_step(state, act)
    
    # left_leg_data = act[jnp.array(left_leg_indices)]
    left_leg_data = act[:3]

    # 添加批次维度以防BatchNorm问题
    left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)

    right_leg_tensor = gnn_model(left_leg_tensor) 
    
    # 确保形状正确
    action = np.concatenate([
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        right_leg_tensor.cpu().detach().numpy().squeeze()
    ])  
    
    # 使用元组而不是列表，并移除多余维度
    data.append((
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        right_leg_tensor.cpu().detach().numpy().squeeze()
    ))
    
    state = jax.jit(env.step)(state, jnp.array(action))
    rollout.append(state.pipeline_state)

print("data:", data[0])  # 只打印第一个样本查看格式

html_output = html.render(env.sys.tree_replace({'opt.timestep': env.dt}), rollout)
with open("output_test_8.html", "w") as f:
    f.write(html_output)
print("Saved visualization as output.html. Open it in a browser.")

In [26]:
class SimpleWalkerGCN(nn.Module):
    """简化的图卷积网络(GCN)模型"""
    def __init__(self, input_dim=1, hidden_dim=16, output_dim=3):
        super(SimpleWalkerGCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)
        
        # 权重初始化
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        # 应用GCN层
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        
        # 每个图在batch中的索引
        batch_idx = data.batch if hasattr(data, 'batch') else None
        
        # 如果是批处理，需要为每个图处理对应节点
        if batch_idx is not None:
            batch_size = batch_idx.max().item() + 1
            outputs = []
            
            for i in range(batch_size):
                # 计算当前图的节点索引基准
                base_idx = i * 6  # 每个图有6个节点
                # 获取右腿节点 (基准索引 + 3, 4, 5)
                right_nodes = x[base_idx + 3:base_idx + 6]
                
                # 改变这里：取平均而不是连接
                combined = torch.mean(right_nodes, dim=0).unsqueeze(0)
                
                # 应用输出层
                out = self.out(combined)
                outputs.append(out)
            
            # 堆叠所有输出
            return torch.cat(outputs, dim=0)
        else:
            # 单个图处理
            right_nodes = x[3:6]
            combined = torch.mean(right_nodes, dim=0).unsqueeze(0)
            return self.out(combined)

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
import time
import os
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.data import Data, Batch

In [28]:
class WalkerGraphBuilder:
    def __init__(self):
        """
        初始化图结构生成器，定义人体腿部关节的连接关系
        """
        # 定义图的结构 - 假设我们有6个节点（左右腿各3个关节）
        self.num_nodes = 6
        
        # 定义边连接 - 按照人体骨骼结构
        self.edge_index_src = [
            0, 1,  # 左腿：大腿→小腿，小腿→脚
            3, 4,  # 右腿：大腿→小腿，小腿→脚
            0, 3,  # 左右腿大腿之间的关系
            1, 4,  # 左右腿小腿之间的关系
            2, 5   # 左右腿脚之间的关系wlak
        ]
        
        # 目标节点
        self.edge_index_dst = [
            1, 2,  # 左腿连接
            4, 5,  # 右腿连接
            3, 0,  # 左右腿大腿连接（双向）
            4, 1,  # 左右腿小腿连接（双向）
            5, 2   # 左右腿脚连接（双向）
        ]
        
        # 转换为PyTorch张量
        self.edge_index = torch.tensor([self.edge_index_src, self.edge_index_dst], 
                                      dtype=torch.long)
    
    def build_graph_from_leg_data(self, left_leg, right_leg):
        """
        将左右腿数据构建为图
        
        参数:
        - left_leg: 左腿3个关节的数据
        - right_leg: 右腿3个关节的数据
        
        返回:
        - torch_geometric.data.Data 对象
        """
        # 确保输入是正确的格式
        if isinstance(left_leg, torch.Tensor):
            left_leg = left_leg.detach().cpu().numpy()
        if isinstance(right_leg, torch.Tensor):
            right_leg = right_leg.detach().cpu().numpy()
            
        # 组合所有节点特征 - 每个节点一个特征维度
        x = torch.cat([
            torch.tensor(left_leg, dtype=torch.float32).view(-1, 1),  # 左腿节点
            torch.tensor(right_leg, dtype=torch.float32).view(-1, 1)  # 右腿节点
        ], dim=0)
        
        # 创建图数据对象
        graph = Data(x=x, edge_index=self.edge_index)
        
        return graph
    
    def build_batch_graphs(self, left_legs, right_legs, device=None):
        """
        批量构建图并合并为一个批次
        
        参数:
        - left_legs: 批量左腿数据 [batch_size, 3]
        - right_legs: 批量右腿数据 [batch_size, 3]
        - device: 设备 (CPU/GPU)
        
        返回:
        - batched_graphs: 批处理后的图数据
        - stacked_targets: 批量右腿标签数据
        """
        batch_size = left_legs.size(0)
        graphs = []
        targets = []
        
        for i in range(batch_size):
            left_leg = left_legs[i]
            right_leg = right_legs[i]
            
            # 构建单个图
            graph = self.build_graph_from_leg_data(left_leg, right_leg)
            graphs.append(graph)
            
            # 保存目标（右腿数据作为标签）
            targets.append(right_leg)
        
        # 合并为批次图
        batched_graphs = Batch.from_data_list(graphs)
        stacked_targets = torch.stack(targets)
        
        # 移动到指定设备
        if device is not None:
            batched_graphs = batched_graphs.to(device)
            stacked_targets = stacked_targets.to(device)
        
        return batched_graphs, stacked_targets


In [29]:
def gnn_inference(model, left_leg_data, graph_builder, device=None, right_leg_dummy=None):
    """
    使用训练好的GNN模型进行推理
    
    参数:
    - model: 训练好的GNN模型
    - left_leg_data: 左腿数据 (3,)
    - graph_builder: 图构建器
    - device: 设备 (CPU/GPU)
    - right_leg_dummy: 占位符，用于构建图 (可选)
    
    返回:
    - 预测的右腿数据 (3,)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    model.eval()
    
    # 确保输入格式正确
    if isinstance(left_leg_data, np.ndarray):
        left_leg_tensor = torch.tensor(left_leg_data, dtype=torch.float32)
    elif isinstance(left_leg_data, torch.Tensor):
        left_leg_tensor = left_leg_data
    else:
        raise TypeError("输入数据类型必须是NumPy数组或PyTorch张量")
    
    # 为构建图，我们需要一个右腿占位符
    if right_leg_dummy is None:
        right_leg_dummy = np.zeros_like(left_leg_tensor.detach().cpu().numpy())
    
    # 构建图
    graph = graph_builder.build_graph_from_leg_data(left_leg_tensor, right_leg_dummy)
    graph = graph.to(device)
    
    # 推理
    with torch.no_grad():
        output = model(graph)
            
    return output.cpu().numpy()

In [30]:
save_dir="walker_gnn_results"
checkpoint = torch.load(os.path.join(save_dir, 'best_model.pth'))
gnn_model = SimpleWalkerGCN(input_dim=1, hidden_dim=16, output_dim=3).to(device)
gnn_model.load_state_dict(checkpoint['model_state_dict'])
graph_builder = WalkerGraphBuilder()
# gnn_model.eval()

In [31]:
#@title Visualizing a trajectory of the learned inference function
import jax
import jax.numpy as jnp
# create an env with auto-reset

env_name = 'walker2d'
backend = 'positional'
env = envs.create(env_name=env_name, backend=backend)

jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
# jit_inference_fn = jax.jit(inference_fn)
data = []
rollout = []
rng = jax.random.PRNGKey(seed=1)
gnn_model.to(device)
gnn_model.eval()
state = jit_env_reset(rng=rng)
for _ in range(1000):
    rollout.append(state.pipeline_state)
    act_rng, rng = jax.random.split(rng)
    act, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_env_step(state, act)
    
    # left_leg_data = act[jnp.array(left_leg_indices)]
    left_leg_data = act[:3]

    # 添加批次维度以防BatchNorm问题
    left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)

    # right_leg_tensor = gnn_model(left_leg_tensor) 
    right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder)
    
    # 确保形状正确
    action = np.concatenate([
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        # right_leg_tensor.cpu().detach().numpy().squeeze()
        np.array(right_leg).squeeze()

    ])  
    
    # 使用元组而不是列表，并移除多余维度
    data.append((
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        # right_leg_tensor.cpu().detach().numpy().squeeze()
        np.array(right_leg).squeeze()
    ))
    
    state = jax.jit(env.step)(state, jnp.array(action))
    rollout.append(state.pipeline_state)

print("data:", data[0])  # 只打印第一个样本查看格式

html_output = html.render(env.sys.tree_replace({'opt.timestep': env.dt}), rollout)
with open("output_test_15.html", "w") as f:
    f.write(html_output)
print("Saved visualization as output.html. Open it in a browser.")

data: (array([-0.888,  0.243,  0.985], dtype=float32), array([-0.035,  0.293,  0.197], dtype=float32))
Saved visualization as output.html. Open it in a browser.


# Temporal GNN

In [9]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
import time
import os

# 导入图神经网络相关库
try:
    from torch_geometric.nn import GCNConv, GATConv, RGCNConv
    from torch_geometric.data import Data, Batch
    # 导入时序GNN处理工具
    from torch_geometric.nn import GatedGraphConv
    # 移除TGCN导入，该模块可能在你的PyTorch Geometric版本中不可用
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    raise ImportError("请安装 PyTorch Geometric 库以使用图神经网络。运行: pip install torch-geometric torch-scatter torch-sparse")

# 设置随机种子以确保结果可重现
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()


class TemporalWalkerGNN(nn.Module):
    """时序图神经网络模型，处理行走时序数据"""
    
    def __init__(self, input_dim=1, hidden_dim=16, output_dim=3, num_nodes=6, sequence_length=5,
                num_relations=2, num_layers=2):
        super(TemporalWalkerGNN, self).__init__()
        
        self.num_nodes = num_nodes
        self.sequence_length = sequence_length
        self.total_nodes = num_nodes * sequence_length
        
        # 使用关系型GCN处理不同类型的边（空间边和时间边）
        self.rgcn1 = RGCNConv(input_dim, hidden_dim, num_relations=num_relations)
        
        # 多层RGCN以增强模型表达能力
        self.rgcn_layers = nn.ModuleList([
            RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
            for _ in range(num_layers - 1)
        ])
        
        # 时间注意力机制 - 在不同时间步之间的注意力
        self.time_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # 最终的输出层
        self.out = nn.Linear(hidden_dim, output_dim)
        
        # 权重初始化
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, data):
        x, edge_index, edge_type = data.x, data.edge_index, data.edge_type
        batch = data.batch if hasattr(data, 'batch') else None
        
        # 应用关系型GCN层
        x = F.relu(self.rgcn1(x, edge_index, edge_type))
        
        # 应用额外的RGCN层
        for layer in self.rgcn_layers:
            x = F.relu(layer(x, edge_index, edge_type))
        
        # 处理批次数据
        if batch is not None:
            batch_size = batch.max().item() + 1
            outputs = []
            
            for i in range(batch_size):
                # 找出当前批次样本的所有节点
                mask = (batch == i)
                batch_x = x[mask]
                
                # 检查并打印张量形状信息
                total_expected_nodes = self.sequence_length * self.num_nodes
                
                # 重塑为 [sequence_length, num_nodes, hidden_dim]
                try:
                    batch_x = batch_x.view(self.sequence_length, self.num_nodes, -1)
                except RuntimeError:
                    # 如果形状不匹配，打印详细信息并尝试修复
                    print(f"Shape error - batch_x: {batch_x.shape}, expected: [{self.sequence_length}, {self.num_nodes}, hidden_dim]")
                    # 确保节点数量是预期的
                    if batch_x.size(0) < total_expected_nodes:
                        # 填充缺失节点
                        pad_size = total_expected_nodes - batch_x.size(0)
                        padding = torch.zeros(pad_size, batch_x.size(1), device=batch_x.device)
                        batch_x = torch.cat([batch_x, padding], dim=0)
                    elif batch_x.size(0) > total_expected_nodes:
                        # 截断多余节点
                        batch_x = batch_x[:total_expected_nodes]
                    
                    batch_x = batch_x.view(self.sequence_length, self.num_nodes, -1)
                
                # 应用时间注意力 - 计算每个时间步的权重
                time_scores = self.time_attention(batch_x.mean(dim=1))  # [sequence_length, 1]
                time_weights = F.softmax(time_scores, dim=0)  # [sequence_length, 1]
                
                # 加权聚合所有时间步 - 修复维度不匹配问题
                time_weights = time_weights.unsqueeze(1)  # [sequence_length, 1, 1]
                weighted_batch = batch_x * time_weights  # 使用广播 [sequence_length, num_nodes, hidden_dim]
                weighted_representation = torch.sum(weighted_batch, dim=0)  # [num_nodes, hidden_dim]
                
                # 获取右腿节点（节点3，4，5）
                right_leg_representation = weighted_representation[3:6]  # [3, hidden_dim]
                
                # 聚合右腿节点特征
                aggregated_right_leg = torch.mean(right_leg_representation, dim=0)  # [hidden_dim]
                
                # 应用输出层
                out = self.out(aggregated_right_leg)
                outputs.append(out)
            
            # 堆叠所有输出
            return torch.stack(outputs)
        else:
            # 单个样本处理
            batch_x = x.view(self.sequence_length, self.num_nodes, -1)
            
            # 应用时间注意力
            time_scores = self.time_attention(batch_x.mean(dim=1))
            time_weights = F.softmax(time_scores, dim=0)
            
            # 加权聚合 - 同样修复维度不匹配问题
            time_weights = time_weights.unsqueeze(1)  # [sequence_length, 1, 1]
            weighted_batch = batch_x * time_weights
            weighted_representation = torch.sum(weighted_batch, dim=0)
            
            # 获取右腿节点
            right_leg_representation = weighted_representation[3:6]
            
            # 聚合
            aggregated_right_leg = torch.mean(right_leg_representation, dim=0)
            
            # 输出
            return self.out(aggregated_right_leg).unsqueeze(0)

In [15]:
class TemporalWalkerGraphBuilder:
    def __init__(self, sequence_length=5):
        """
        初始化时序图结构生成器，定义人体腿部关节的连接关系
        
        参数:
        - sequence_length: 时间序列长度，表示考虑多少个历史时间步
        """
        # 定义图的结构 - 假设我们有6个节点（左右腿各3个关节）
        self.num_nodes = 6
        self.sequence_length = sequence_length
        
        # 定义空间边连接 - 按照人体骨骼结构
        self.spatial_edge_index_src = [
            0, 1,  # 左腿：大腿→小腿，小腿→脚
            3, 4,  # 右腿：大腿→小腿，小腿→脚
            0, 3,  # 左右腿大腿之间的关系
            1, 4,  # 左右腿小腿之间的关系
            2, 5   # 左右腿脚之间的关系
        ]
        
        # 目标节点
        self.spatial_edge_index_dst = [
            1, 2,  # 左腿连接
            4, 5,  # 右腿连接
            3, 0,  # 左右腿大腿连接（双向）
            4, 1,  # 左右腿小腿连接（双向）
            5, 2   # 左右腿脚连接（双向）
        ]
        
        # 转换为PyTorch张量 - 空间边
        self.spatial_edge_index = torch.tensor([self.spatial_edge_index_src, self.spatial_edge_index_dst], 
                                           dtype=torch.long)
        
        # 创建时间边 - 连接不同时间步的相同节点
        self._build_temporal_edges()
    
    def _build_temporal_edges(self):
        """构建时间边连接不同时间步的相同节点"""
        temporal_edges_src = []
        temporal_edges_dst = []
        
        # 对于每个时间步t和每个节点，连接到t+1时间步的相同节点
        for t in range(self.sequence_length - 1):
            base_t = t * self.num_nodes  # 当前时间步的基础索引
            base_t_next = (t + 1) * self.num_nodes  # 下一个时间步的基础索引
            
            for node in range(self.num_nodes):
                # t时间步的节点连接到t+1时间步的相同节点
                temporal_edges_src.append(base_t + node)
                temporal_edges_dst.append(base_t_next + node)
                
                # 可选：添加更多时间边连接，例如连接到t+2, t+3等
                # 这里只连接到t+1
        
        self.temporal_edge_index = torch.tensor([temporal_edges_src, temporal_edges_dst], 
                                               dtype=torch.long)
        
        # 边的类型: 0=空间边, 1=时间边
        self.edge_type = torch.zeros(len(self.spatial_edge_index_src) + len(temporal_edges_src), 
                                    dtype=torch.long)
        self.edge_type[len(self.spatial_edge_index_src):] = 1
        
        # 组合空间和时间边
        self.combined_edge_index = torch.cat([
            self.spatial_edge_index, 
            self.temporal_edge_index
        ], dim=1)
    
    def build_temporal_graph_from_sequence(self, left_leg_sequence, right_leg_sequence):
        """
        将左右腿的时间序列数据构建为时序图
        
        参数:
        - left_leg_sequence: 左腿序列数据 [seq_len, 3]
        - right_leg_sequence: 右腿序列数据 [seq_len, 3]
        
        返回:
        - torch_geometric.data.Data 对象
        """
        # 确保输入是正确的格式和长度
        if isinstance(left_leg_sequence, torch.Tensor):
            left_leg_sequence = left_leg_sequence.detach().cpu().numpy()
        if isinstance(right_leg_sequence, torch.Tensor):
            right_leg_sequence = right_leg_sequence.detach().cpu().numpy()
            
        # 截断或填充序列以匹配所需的序列长度
        seq_len = min(len(left_leg_sequence), self.sequence_length)
        
        # 准备节点特征 - 每个时间步的每个节点一个特征
        node_features = []
        
        for t in range(seq_len):
            # 获取当前时间步的左右腿数据
            left_leg_t = left_leg_sequence[t]
            right_leg_t = right_leg_sequence[t]
            
            # 添加当前时间步的节点特征
            node_features.append(torch.tensor(left_leg_t, dtype=torch.float32).view(-1, 1))
            node_features.append(torch.tensor(right_leg_t, dtype=torch.float32).view(-1, 1))
        
        # 如果序列长度小于所需长度，用最后一个时间步的数据填充
        if seq_len < self.sequence_length:
            last_left = left_leg_sequence[-1]
            last_right = right_leg_sequence[-1]
            
            for _ in range(self.sequence_length - seq_len):
                node_features.append(torch.tensor(last_left, dtype=torch.float32).view(-1, 1))
                node_features.append(torch.tensor(last_right, dtype=torch.float32).view(-1, 1))
        
        # 合并所有节点特征
        x = torch.cat(node_features, dim=0)
        
        # 创建图数据对象，包括时间信息
        graph = Data(
            x=x, 
            edge_index=self.combined_edge_index,
            edge_type=self.edge_type,
            num_nodes=x.size(0)
        )
        
        return graph
    
    def build_batch_temporal_graphs(self, left_leg_sequences, right_leg_sequences, device=None):
        """
        批量构建时序图并合并为一个批次
        
        参数:
        - left_leg_sequences: 批量左腿序列数据 [batch_size, seq_len, 3]
        - right_leg_sequences: 批量右腿序列数据 [batch_size, seq_len, 3]
        - device: 设备 (CPU/GPU)
        
        返回:
        - batched_graphs: 批处理后的图数据
        - stacked_targets: 批量右腿标签数据（最后一个时间步）
        """
        batch_size = left_leg_sequences.size(0)
        graphs = []
        targets = []
        
        for i in range(batch_size):
            left_seq = left_leg_sequences[i]
            right_seq = right_leg_sequences[i]
            
            # 构建单个时序图
            graph = self.build_temporal_graph_from_sequence(left_seq, right_seq)
            graphs.append(graph)
            
            # 保存目标（右腿序列最后一个时间步作为标签）
            if isinstance(right_seq, torch.Tensor):
                targets.append(right_seq[-1])
            else:
                targets.append(torch.tensor(right_seq[-1], dtype=torch.float32))
        
        # 合并为批次图
        batched_graphs = Batch.from_data_list(graphs)
        stacked_targets = torch.stack(targets)
        
        # 移动到指定设备
        if device is not None:
            batched_graphs = batched_graphs.to(device)
            stacked_targets = stacked_targets.to(device)
        
        return batched_graphs, stacked_targets

In [17]:
def temporal_gnn_inference(model, left_leg_sequence, graph_builder, device=None, right_leg_dummy=None):
    """
    使用训练好的时序GNN模型进行推理
    
    参数:
    - model: 训练好的时序GNN模型
    - left_leg_sequence: 左腿序列数据 [seq_len, 3]
    - graph_builder: 时序图构建器
    - device: 设备 (CPU/GPU)
    - right_leg_dummy: 占位符，用于构建图
    
    返回:
    - 预测的右腿数据 (3,)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    model.eval()
    
    # 确保输入格式正确
    if isinstance(left_leg_sequence, np.ndarray):
        left_seq_tensor = torch.tensor(left_leg_sequence, dtype=torch.float32)
    elif isinstance(left_leg_sequence, torch.Tensor):
        left_seq_tensor = left_leg_sequence
    else:
        raise TypeError("输入数据类型必须是NumPy数组或PyTorch张量")
    
    # 为构建图，我们需要一个右腿占位符序列
    if right_leg_dummy is None:
        if isinstance(left_leg_sequence, np.ndarray):
            right_leg_dummy = np.zeros_like(left_leg_sequence)
        else:
            right_leg_dummy = torch.zeros_like(left_seq_tensor)
    
    # 构建时序图
    graph = graph_builder.build_temporal_graph_from_sequence(left_seq_tensor, right_leg_dummy)
    graph = graph.to(device)
    
    # 推理
    with torch.no_grad():
        output = model(graph)
            
    return output.cpu().numpy()


In [19]:
hidden_dim=16
sequence_length=5
gnn_temporal_model = TemporalWalkerGNN(
        input_dim=1, 
        hidden_dim=hidden_dim, 
        output_dim=3, 
        num_nodes=6, 
        sequence_length=sequence_length,
        num_relations=2,  # 空间关系和时间关系
        num_layers=2
    ).to(device)

In [20]:
save_dir="/u20/li3658/rl_training/temporal_walker_gnn_results"
checkpoint = torch.load(os.path.join(save_dir, 'best_model.pth'))
gnn_temporal_model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [21]:
graph_temporal_builder = TemporalWalkerGraphBuilder(sequence_length=5)

In [23]:
#@title Visualizing a trajectory of the learned inference function
import jax
import jax.numpy as jnp
# create an env with auto-reset

env_name = 'walker2d'
backend = 'positional'
env = envs.create(env_name=env_name, backend=backend)

jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
# jit_inference_fn = jax.jit(inference_fn)
data = []
rollout = []
rng = jax.random.PRNGKey(seed=1)
gnn_temporal_model.to(device)
gnn_temporal_model.eval()
state = jit_env_reset(rng=rng)
for _ in range(1000):
    rollout.append(state.pipeline_state)
    act_rng, rng = jax.random.split(rng)
    act, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_env_step(state, act)
    
    # left_leg_data = act[jnp.array(left_leg_indices)]
    left_leg_data = act[:3]

    # 添加批次维度以防BatchNorm问题
    left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)

    # right_leg_tensor = gnn_model(left_leg_tensor) 
    # right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder)
    predicted_right_leg = temporal_gnn_inference(gnn_temporal_model, left_leg_tensor, graph_builder)
    
    # 确保形状正确
    action = np.concatenate([
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        # right_leg_tensor.cpu().detach().numpy().squeeze()
        np.array(predicted_right_leg).squeeze()

    ])  
    
    # 使用元组而不是列表，并移除多余维度
    data.append((
        left_leg_tensor.cpu().detach().numpy().squeeze(), 
        # right_leg_tensor.cpu().detach().numpy().squeeze()
        np.array(predicted_right_leg).squeeze()
    ))
    
    state = jax.jit(env.step)(state, jnp.array(action))
    rollout.append(state.pipeline_state)

print("data:", data[0])  # 只打印第一个样本查看格式

html_output = html.render(env.sys.tree_replace({'opt.timestep': env.dt}), rollout)
with open("output_test_21.html", "w") as f:
    f.write(html_output)
print("Saved visualization as output.html. Open it in a browser.")

data: (array([-0.888,  0.243,  0.985], dtype=float32), array([ 0.39 , -0.21 , -0.158], dtype=float32))
Saved visualization as output.html. Open it in a browser.


In [49]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
# import html
# from brax import envs

def multi_walker_visualization(gnn_model, graph_builder, n_walkers=2, n_steps=1000, device='cuda'):
    """
    为多个Walker2D生成可视化轨迹
    
    参数:
    - gnn_model: 预训练的GNN模型
    - graph_builder: WalkerGraphBuilder实例
    - n_walkers: Walker数量
    - n_steps: 模拟步数
    - device: 计算设备
    
    返回:
    - 可视化HTML输出
    """
    # 创建多个Walker2D环境
    env_name = 'walker2d'
    backend = 'positional'
    walker_envs = []
    walker_resets = []
    walker_steps = []
    
    for _ in range(n_walkers):
        env = envs.create(env_name=env_name, backend=backend)
        walker_envs.append(env)
        walker_resets.append(jax.jit(env.reset))
        walker_steps.append(jax.jit(env.step))
    
    # 获取RL推理函数（假设已经定义）
    # from your_rl_model import inference_fn  # 替换为您的实际导入
    # jit_inference_fn = jax.jit(inference_fn)
    
    # 准备GNN模型
    gnn_model.to(device)
    gnn_model.eval()
    
    # 初始化随机种子
    base_rng = jax.random.PRNGKey(seed=1)
    rngs = []
    for i in range(n_walkers):
        base_rng, sub_rng = jax.random.split(base_rng)
        rngs.append(sub_rng)
    
    # 初始化环境状态
    states = []
    for i in range(n_walkers):
        states.append(walker_resets[i](rng=rngs[i]))
    
    # 数据收集
    all_data = [[] for _ in range(n_walkers)]
    all_rollouts = [[] for _ in range(n_walkers)]
    
    # 辅助函数：进行GNN推理
    # def gnn_inference(model, left_leg_data, graph_builder, device):
    #     """使用GNN模型预测右腿动作"""
    #     # 确保输入是PyTorch张量
    #     if isinstance(left_leg_data, np.ndarray):
    #         left_leg_tensor = torch.tensor(left_leg_data, dtype=torch.float32)
    #     elif 'jax' in str(type(left_leg_data)):
    #         left_leg_tensor = torch.tensor(np.array(left_leg_data), dtype=torch.float32)
    #     else:
    #         left_leg_tensor = left_leg_data
            
    #     # 移至设备
    #     left_leg_tensor = left_leg_tensor.to(device)
        
    #     # 创建右腿占位符
    #     right_leg_dummy = torch.zeros_like(left_leg_tensor).to(device)
        
    #     # 构建图
    #     graph = graph_builder.build_graph_from_leg_data(left_leg_tensor.cpu(), right_leg_dummy.cpu())
    #     graph = graph.to(device)
        
    #     # 推理
    #     with torch.no_grad():
    #         output = model(graph)
            
    #     # 返回结果
    #     return output.cpu().numpy()
    
    # 计算跟随策略 - 为跟随者生成动作
    def compute_follower_action(leader_action, previous_follower_action=None, follow_factor=0.8):
        """计算跟随者的动作，基于领导者的动作和之前自己的动作"""
        # 如果没有之前的动作，完全跟随领导者
        if previous_follower_action is None:
            # 添加轻微延迟（少量随机噪声）
            noise = np.random.normal(0, 0.05, leader_action.shape)
            return np.array(leader_action) + noise
        
        # 否则，部分跟随领导者，部分保持自己的动作连续性
        follower_action = follow_factor * np.array(leader_action) + \
                          (1 - follow_factor) * previous_follower_action
        
        # 添加少量随机性
        noise = np.random.normal(0, 0.02, leader_action.shape)
        
        return follower_action + noise
    
    # 保存上一步的跟随者动作
    previous_follower_actions = [None for _ in range(n_walkers-1)]
    
    # 模拟循环
    for step in range(n_steps):
        # 记录所有Walker的当前状态
        for i in range(n_walkers):
            all_rollouts[i].append(states[i].pipeline_state)
        
        # 为领导者生成动作（使用RL策略）
        rngs[0], act_rng = jax.random.split(rngs[0])
        leader_action, _ = jit_inference_fn(states[0].obs, act_rng)
        
        # 提取领导者的左腿动作
        leader_left_leg = leader_action[:3]
        
        # 使用GNN预测领导者的右腿动作
        leader_right_leg = gnn_inference(gnn_model, leader_left_leg, graph_builder, device)
        
        # 构建领导者的完整动作
        leader_full_action = np.concatenate([
            np.array(leader_left_leg),
            leader_right_leg.squeeze()
        ])
        
        # 为所有Walker准备动作
        actions = [leader_full_action]
        
        # 为跟随者生成动作
        for i in range(1, n_walkers):
            # 计算跟随者的左腿动作
            follower_action = compute_follower_action(
                leader_action,
                previous_follower_actions[i-1],
                follow_factor=0.9 - 0.1 * (i-1)  # 跟随系数随距离减小
            )
            
            # 提取跟随者的左腿动作
            follower_left_leg = follower_action[:3]
            
            # 使用GNN预测跟随者的右腿动作
            follower_right_leg = gnn_inference(gnn_model, follower_left_leg, graph_builder, device)
            
            # 构建跟随者的完整动作
            follower_full_action = np.concatenate([
                follower_left_leg,
                follower_right_leg.squeeze()
            ])
            
            # 保存动作供下一步使用
            previous_follower_actions[i-1] = follower_action
            
            # 添加到动作列表
            actions.append(follower_full_action)
        
        # 收集数据
        for i in range(n_walkers):
            left_leg_data = actions[i][:3]
            right_leg_data = actions[i][3:]
            
            all_data[i].append((left_leg_data, right_leg_data))
        
        # 环境步进
        new_states = []
        for i in range(n_walkers):
            # 转换动作类型
            if 'jax' in str(type(states[i])):
                jax_action = jnp.array(actions[i])
                new_state = walker_steps[i](states[i], jax_action)
            else:
                new_state = walker_steps[i](states[i], actions[i])
                
            new_states.append(new_state)
            all_rollouts[i].append(new_state.pipeline_state)
        
        # 更新状态
        states = new_states
    
    # 打印数据样本
    for i in range(n_walkers):
        print(f"Walker {i+1} 数据示例:", all_data[i][0])
    
    # 生成可视化 - 每个Walker一个单独的HTML文件
    for i in range(n_walkers):
        html_output = html.render(walker_envs[i].sys.tree_replace(
            {'opt.timestep': walker_envs[i].dt}), all_rollouts[i])
        
        with open(f"walker_{i+1}_output.html", "w") as f:
            f.write(html_output)
        
        print(f"Saved visualization for Walker {i+1} as walker_{i+1}_output.html")

    # 返回所有收集的数据
    return all_data, all_rollouts


# 主函数：运行多Walker模拟
if __name__ == "__main__":
    # 导入您的模型和图构建器
    # from your_model_file import SimpleWalkerGCN, WalkerGraphBuilder
    import torch
    import os
    
    # 加载GNN模型
    save_dir = "walker_gnn_results"
    checkpoint = torch.load(os.path.join(save_dir, "best_model.pth"))
    
    gnn_model = SimpleWalkerGCN(input_dim=1, hidden_dim=16, output_dim=3)
    gnn_model.load_state_dict(checkpoint['model_state_dict'])
    
    # 创建图构建器
    graph_builder = WalkerGraphBuilder()
    
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 运行多Walker模拟
    n_walkers = 2  # 设置Walker数量
    data, rollouts = multi_walker_visualization(
        gnn_model=gnn_model,
        graph_builder=graph_builder,
        n_walkers=n_walkers,
        device=device
    )
    
    print("多Walker2D模拟完成！")

Walker 1 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Walker 2 数据示例: (array([-0.999,  0.403,  0.796], dtype=float32), array([-0.063,  0.16 ,  0.041], dtype=float32))
Saved visualization for Walker 1 as walker_1_output.html
Saved visualization for Walker 2 as walker_2_output.html
多Walker2D模拟完成！


In [53]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
# import html
# from brax import envs

def multi_walker_visualization(gnn_model, graph_builder, jit_inference_fn, n_walkers=2, n_steps=1000, device='cuda'):
    """
    为多个Walker2D生成可视化轨迹
    
    参数:
    - gnn_model: 预训练的GNN模型
    - graph_builder: WalkerGraphBuilder实例
    - jit_inference_fn: JAX JIT编译的RL推理函数
    - n_walkers: Walker数量
    - n_steps: 模拟步数
    - device: 计算设备
    
    返回:
    - 可视化HTML输出
    """
    # 创建多个Walker2D环境
    env_name = 'walker2d'
    backend = 'positional'
    walker_envs = []
    walker_resets = []
    walker_steps = []
    
    for _ in range(n_walkers):
        env = envs.create(env_name=env_name, backend=backend)
        walker_envs.append(env)
        walker_resets.append(jax.jit(env.reset))
        walker_steps.append(jax.jit(env.step))
    
    # 准备GNN模型
    gnn_model.to(device)
    gnn_model.eval()
    
    # 初始化随机种子
    base_rng = jax.random.PRNGKey(seed=1)
    rngs = []
    for i in range(n_walkers):
        base_rng, sub_rng = jax.random.split(base_rng)
        rngs.append(sub_rng)
    
    # 初始化环境状态
    states = []
    for i in range(n_walkers):
        states.append(walker_resets[i](rng=rngs[i]))
    
    # 数据收集
    all_data = [[] for _ in range(n_walkers)]
    all_rollouts = [[] for _ in range(n_walkers)]
    
    # # 辅助函数：进行GNN推理 - 保持与原始代码一致
    # def gnn_inference(model, left_leg_data, graph_builder, device):
    #     """使用GNN模型预测右腿动作"""
    #     # 确保输入是PyTorch张量并增加批次维度
    #     if isinstance(left_leg_data, np.ndarray):
    #         left_leg_tensor = torch.tensor(left_leg_data, dtype=torch.float32)
    #     elif 'jax' in str(type(left_leg_data)):
    #         left_leg_tensor = torch.tensor(np.array(left_leg_data), dtype=torch.float32)
    #     else:
    #         left_leg_tensor = left_leg_data
            
    #     # 移至设备并确保有批次维度
    #     if len(left_leg_tensor.shape) == 1:
    #         left_leg_tensor = left_leg_tensor.unsqueeze(0)
    #     left_leg_tensor = left_leg_tensor.to(device)
        
    #     # 创建右腿占位符
    #     right_leg_dummy = torch.zeros_like(left_leg_tensor).to(device)
        
    #     # 构建图
    #     graph = graph_builder.build_graph_from_leg_data(left_leg_tensor.cpu(), right_leg_dummy.cpu())
    #     graph = graph.to(device)
        
    #     # 推理
    #     with torch.no_grad():
    #         output = model(graph)
            
    #     # 返回结果
    #     return output.cpu().numpy()
    
    # 模拟循环
    for step in range(n_steps):
        # 处理每个Walker
        for i in range(n_walkers):
            # 记录当前状态
            all_rollouts[i].append(states[i].pipeline_state)
            
            # 生成动作 - 按照与单Walker相同的方式
            if i == 0:  # 领导者使用RL策略
                rngs[i], act_rng = jax.random.split(rngs[i])
                act, _ = jit_inference_fn(states[i].obs, act_rng)
                
                # 重要：使用RL动作先更新一次状态
                states[i] = walker_steps[i](states[i], act)
                
                # 提取左腿动作
                left_leg_data = act[:3]
                
                # GNN预测右腿动作
                left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                
                # 创建完整动作
                action = np.concatenate([
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ])
                
                # 再次更新状态 - 使用完整动作
                states[i] = walker_steps[i](states[i], jnp.array(action))
                
                # 记录数据
                all_data[i].append((
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
                
                # 保存领导者动作供跟随者使用
                leader_action = act
                # leader_position = np.array(states[i].pipeline_state.qp.pos[0])
            
            else:  # 跟随者
                # 为跟随者生成修改后的动作
                follower_offset = i * 1.0  # 跟随距离
                position_diff = np.array([follower_offset, 0.0, 0.0])  # x方向间隔
                
                # 跟随者观察领导者的动作，但添加延迟/变异
                noise_scale = 0.05 * i  # 随距离增加噪声
                # follower_action = np.array(leader_action) + np.random.normal(0, noise_scale, leader_action.shape)
                follower_action, _ = jit_inference_fn(states[i].obs, act_rng)

                
                # 先使用基本动作更新状态
                states[i] = walker_steps[i](states[i], jnp.array(follower_action))
                
                # 提取左腿动作
                left_leg_data = follower_action[:3]
                
                # GNN预测右腿动作
                left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                
                # 创建完整动作
                action = np.concatenate([
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ])
                
                # 再次更新状态 - 使用完整动作
                states[i] = walker_steps[i](states[i], jnp.array(action))
                
                # 记录数据
                all_data[i].append((
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
            
            # 调试输出 - 每100步检查一次位置
            if step % 100 == 0:
                try:
                    position = np.array(states[i].pipeline_state.qp.pos[0])
                    print(f"Step {step}, Walker {i+1} position: {position}")
                except:
                    print(f"Step {step}, Walker {i+1} 无法获取位置")
    
    # 打印数据样本
    for i in range(n_walkers):
        print(f"Walker {i+1} 数据示例:", all_data[i][0])
    
    # 生成可视化 - 每个Walker一个单独的HTML文件
    for i in range(n_walkers):
        html_output = html.render(walker_envs[i].sys.tree_replace(
            {'opt.timestep': walker_envs[i].dt}), all_rollouts[i])
        
        with open(f"walker_{i+1}_output.html", "w") as f:
            f.write(html_output)
        
        print(f"Saved visualization for Walker {i+1} as walker_{i+1}_output.html")

    # 返回所有收集的数据
    return all_data, all_rollouts


# 主函数：运行多Walker模拟
if __name__ == "__main__":
    # 导入您的模型和图构建器
    # from your_model_file import SimpleWalkerGCN, WalkerGraphBuilder
    import torch
    import os
    
    # 加载GNN模型
    save_dir = "walker_gnn_results"
    checkpoint = torch.load(os.path.join(save_dir, "best_model.pth"))
    
    gnn_model = SimpleWalkerGCN(input_dim=1, hidden_dim=16, output_dim=3)
    gnn_model.load_state_dict(checkpoint['model_state_dict'])
    
    # 创建图构建器
    graph_builder = WalkerGraphBuilder()
    
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 获取RL推理函数
    # jit_inference_fn = jax.jit(inference_fn)  # 确保这个已定义
    
    # 运行多Walker模拟
    n_walkers = 2  # 设置Walker数量
    data, rollouts = multi_walker_visualization(
        gnn_model=gnn_model,
        graph_builder=graph_builder,
        jit_inference_fn=jit_inference_fn,
        n_walkers=n_walkers,
        device=device
    )
    
    print("多Walker2D模拟完成！")

Step 0, Walker 1 无法获取位置
Step 0, Walker 2 无法获取位置
Step 100, Walker 1 无法获取位置
Step 100, Walker 2 无法获取位置
Step 200, Walker 1 无法获取位置
Step 200, Walker 2 无法获取位置
Step 300, Walker 1 无法获取位置
Step 300, Walker 2 无法获取位置
Step 400, Walker 1 无法获取位置
Step 400, Walker 2 无法获取位置
Step 500, Walker 1 无法获取位置
Step 500, Walker 2 无法获取位置
Step 600, Walker 1 无法获取位置
Step 600, Walker 2 无法获取位置
Step 700, Walker 1 无法获取位置
Step 700, Walker 2 无法获取位置
Step 800, Walker 1 无法获取位置
Step 800, Walker 2 无法获取位置
Step 900, Walker 1 无法获取位置
Step 900, Walker 2 无法获取位置
Walker 1 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Walker 2 数据示例: (array([-0.963,  0.511,  0.925], dtype=float32), array([0.048, 0.149, 0.414], dtype=float32))
Saved visualization for Walker 1 as walker_1_output.html
Saved visualization for Walker 2 as walker_2_output.html
多Walker2D模拟完成！


In [51]:
import jax
import jax.numpy as jnp
import numpy as np
import torch
# import html
# from brax import envs

def multi_walker_visualization(gnn_model, graph_builder, jit_inference_fn, n_walkers=2, n_steps=1000, device='cuda'):
    """
    为多个Walker2D生成可视化轨迹
    
    参数:
    - gnn_model: 预训练的GNN模型
    - graph_builder: WalkerGraphBuilder实例
    - jit_inference_fn: JAX JIT编译的RL推理函数
    - n_walkers: Walker数量
    - n_steps: 模拟步数
    - device: 计算设备
    
    返回:
    - 可视化HTML输出
    """
    # 创建多个Walker2D环境
    env_name = 'walker2d'
    backend = 'positional'
    walker_envs = []
    walker_resets = []
    walker_steps = []
    
    for _ in range(n_walkers):
        env = envs.create(env_name=env_name, backend=backend)
        walker_envs.append(env)
        walker_resets.append(jax.jit(env.reset))
        walker_steps.append(jax.jit(env.step))
    
    # 准备GNN模型
    gnn_model.to(device)
    gnn_model.eval()
    
    # 初始化随机种子
    base_rng = jax.random.PRNGKey(seed=1)
    rngs = []
    for i in range(n_walkers):
        base_rng, sub_rng = jax.random.split(base_rng)
        rngs.append(sub_rng)
    
    # 初始化环境状态
    states = []
    for i in range(n_walkers):
        states.append(walker_resets[i](rng=rngs[i]))
    
    # 数据收集
    all_data = [[] for _ in range(n_walkers)]
    all_rollouts = [[] for _ in range(n_walkers)]
    
    # 存储跟随者的最近动作 - 用于持续的GNN预测
    follower_last_actions = [None] * (n_walkers - 1)
    
    # 辅助函数：进行GNN推理
    def gnn_inference(model, left_leg_data, graph_builder, device):
        """使用GNN模型预测右腿动作"""
        # 确保输入是PyTorch张量并增加批次维度
        if isinstance(left_leg_data, np.ndarray):
            left_leg_tensor = torch.tensor(left_leg_data, dtype=torch.float32)
        elif 'jax' in str(type(left_leg_data)):
            left_leg_tensor = torch.tensor(np.array(left_leg_data), dtype=torch.float32)
        else:
            left_leg_tensor = left_leg_data
            
        # 移至设备并确保有批次维度
        if len(left_leg_tensor.shape) == 1:
            left_leg_tensor = left_leg_tensor.unsqueeze(0)
        left_leg_tensor = left_leg_tensor.to(device)
        
        # 创建右腿占位符
        right_leg_dummy = torch.zeros_like(left_leg_tensor).to(device)
        
        # 构建图
        graph = graph_builder.build_graph_from_leg_data(left_leg_tensor.cpu(), right_leg_dummy.cpu())
        graph = graph.to(device)
        
        # 推理
        with torch.no_grad():
            output = model(graph)
            
        # 返回结果
        return output.cpu().numpy()
    
    # 辅助函数：预测左腿动作基于右腿
    def predict_left_leg(model, right_leg_data, graph_builder, device):
        """使用GNN模型预测左腿动作，基于右腿数据
        
        注意：这假设模型能够双向工作，从右腿预测左腿"""
        # 构建特殊的图结构，交换左右腿位置
        if isinstance(right_leg_data, np.ndarray):
            right_leg_tensor = torch.tensor(right_leg_data, dtype=torch.float32)
        elif 'jax' in str(type(right_leg_data)):
            right_leg_tensor = torch.tensor(np.array(right_leg_data), dtype=torch.float32)
        else:
            right_leg_tensor = right_leg_data
            
        # 移至设备并确保有批次维度
        if len(right_leg_tensor.shape) == 1:
            right_leg_tensor = right_leg_tensor.unsqueeze(0)
        right_leg_tensor = right_leg_tensor.to(device)
        
        # 创建左腿占位符
        left_leg_dummy = torch.zeros_like(right_leg_tensor).to(device)
        
        # 注意：这里我们需要交换左右腿的位置，让GNN预测左腿
        # 这假设图构建器能处理这种情况，或者我们需要用另一种方式处理
        
        # 这里使用一个简单的方法：使用GNN从右腿推断左腿
        # 实际应用中，您可能需要训练一个专门的模型或使用更复杂的方法
        
        # 简单实现：使用对称性 - 左右腿动作关于中轴镜像
        # 注意：这是一个简化假设，真实情况可能更复杂
        left_leg_prediction = -1.0 * right_leg_tensor.cpu().numpy()
        
        return left_leg_prediction
    
    # 模拟循环
    for step in range(n_steps):
        # 处理每个Walker
        for i in range(n_walkers):
            # 记录当前状态
            all_rollouts[i].append(states[i].pipeline_state)
            
            # 生成动作 - 按照与单Walker相同的方式
            if i == 0:  # 领导者使用RL策略
                rngs[i], act_rng = jax.random.split(rngs[i])
                act, _ = jit_inference_fn(states[i].obs, act_rng)
                
                # 重要：使用RL动作先更新一次状态
                states[i] = walker_steps[i](states[i], act)
                
                # 提取左腿动作
                left_leg_data = act[:3]
                
                # GNN预测右腿动作
                left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                
                # 创建完整动作
                action = np.concatenate([
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ])
                
                # 再次更新状态 - 使用完整动作
                states[i] = walker_steps[i](states[i], jnp.array(action))
                
                # 记录数据
                all_data[i].append((
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
                
                # 保存领导者第一步的动作供跟随者初始化
                if step == 0:
                    leader_first_action = np.array(act)
            
            else:  # 跟随者 - 自主行走
                follower_idx = i - 1  # 跟随者索引
                
                if step == 0:
                    # 第一步：使用领导者的初始动作
                    initial_action = np.array(leader_first_action)
                    
                    # 先使用基本动作更新状态
                    states[i] = walker_steps[i](states[i], jnp.array(initial_action))
                    
                    # 提取左腿动作
                    left_leg_data = initial_action[:3]
                    
                    # GNN预测右腿动作
                    left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                    right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                    
                    # 创建完整动作
                    action = np.concatenate([
                        left_leg_tensor.cpu().detach().numpy().squeeze(),
                        np.array(right_leg).squeeze()
                    ])
                    
                    # 再次更新状态 - 使用完整动作
                    states[i] = walker_steps[i](states[i], jnp.array(action))
                    
                    # 保存该动作用于下一步
                    follower_last_actions[follower_idx] = action.copy()
                
                else:
                    # 后续步骤：使用GNN预测的动作自主行走
                    
                    # 获取上一步的动作
                    last_action = follower_last_actions[follower_idx]
                    
                    # 提取上一步的右腿动作
                    right_leg_data = last_action[3:]
                    
                    # 基于右腿动作预测左腿动作
                    left_leg = predict_left_leg(gnn_model, right_leg_data, graph_builder, device)
                    
                    # 创建包含左腿的临时动作
                    temp_action = np.concatenate([
                        np.array(left_leg).squeeze(),
                        np.zeros(3)  # 临时右腿占位符
                    ])
                    
                    # 先更新状态
                    states[i] = walker_steps[i](states[i], jnp.array(temp_action))
                    
                    # 提取左腿作为输入
                    left_leg_data = left_leg.squeeze()
                    
                    # 使用GNN预测右腿动作
                    left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                    right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                    
                    # 创建完整动作
                    action = np.concatenate([
                        np.array(left_leg).squeeze(),
                        np.array(right_leg).squeeze()
                    ])
                    
                    # 再次更新状态 - 使用完整动作
                    states[i] = walker_steps[i](states[i], jnp.array(action))
                    
                    # 保存该动作用于下一步
                    follower_last_actions[follower_idx] = action.copy()
                
                # 记录数据
                all_data[i].append((
                    np.array(left_leg).squeeze() if step > 0 else left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
            
            # 调试输出 - 每100步检查一次位置
            if step % 100 == 0:
                try:
                    position = np.array(states[i].pipeline_state.qp.pos[0])
                    print(f"Step {step}, Walker {i+1} position: {position}")
                except:
                    print(f"Step {step}, Walker {i+1} 无法获取位置")
    
    # 打印数据样本
    for i in range(n_walkers):
        print(f"Walker {i+1} 数据示例:", all_data[i][0])
    
    # 生成可视化 - 每个Walker一个单独的HTML文件
    for i in range(n_walkers):
        html_output = html.render(walker_envs[i].sys.tree_replace(
            {'opt.timestep': walker_envs[i].dt}), all_rollouts[i])
        
        with open(f"walker_{i+1}_output.html", "w") as f:
            f.write(html_output)
        
        print(f"Saved visualization for Walker {i+1} as walker_{i+1}_output.html")

    # 返回所有收集的数据
    return all_data, all_rollouts


# 实用函数：打开多个浏览器窗口显示
def open_multiple_visualizations(n_walkers):
    """打开多个浏览器窗口显示不同Walker的HTML"""
    import webbrowser
    import time
    
    for i in range(n_walkers):
        filename = f"walker_{i+1}_output.html"
        try:
            webbrowser.open(filename)
            time.sleep(0.5)  # 等待浏览器加载
        except Exception as e:
            print(f"无法打开{filename}：{e}")
    
    print("已打开所有Walker可视化")


# 主函数：运行多Walker模拟
if __name__ == "__main__":
    # 导入您的模型和图构建器
    # from your_model_file import SimpleWalkerGCN, WalkerGraphBuilder
    import torch
    import os
    
    # 加载GNN模型
    save_dir = "walker_gnn_results"
    checkpoint = torch.load(os.path.join(save_dir, "best_model.pth"))
    
    gnn_model = SimpleWalkerGCN(input_dim=1, hidden_dim=16, output_dim=3)
    gnn_model.load_state_dict(checkpoint['model_state_dict'])
    
    # 创建图构建器
    graph_builder = WalkerGraphBuilder()
    
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 获取RL推理函数
    # jit_inference_fn = jax.jit(inference_fn)  # 确保这个已定义
    
    # 运行多Walker模拟
    n_walkers = 2  # 设置Walker数量
    data, rollouts = multi_walker_visualization(
        gnn_model=gnn_model,
        graph_builder=graph_builder,
        jit_inference_fn=jit_inference_fn,
        n_walkers=n_walkers,
        device=device
    )
    
    # 打开所有Walker的可视化
    open_multiple_visualizations(n_walkers)
    
    print("多Walker2D模拟完成！")

Step 0, Walker 1 无法获取位置
Step 0, Walker 2 无法获取位置
Step 100, Walker 1 无法获取位置
Step 100, Walker 2 无法获取位置
Step 200, Walker 1 无法获取位置
Step 200, Walker 2 无法获取位置
Step 300, Walker 1 无法获取位置
Step 300, Walker 2 无法获取位置
Step 400, Walker 1 无法获取位置
Step 400, Walker 2 无法获取位置
Step 500, Walker 1 无法获取位置
Step 500, Walker 2 无法获取位置
Step 600, Walker 1 无法获取位置
Step 600, Walker 2 无法获取位置
Step 700, Walker 1 无法获取位置
Step 700, Walker 2 无法获取位置
Step 800, Walker 1 无法获取位置
Step 800, Walker 2 无法获取位置
Step 900, Walker 1 无法获取位置
Step 900, Walker 2 无法获取位置
Walker 1 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Walker 2 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Saved visualization for Walker 1 as walker_1_output.html
Saved visualization for Walker 2 as walker_2_output.html
已打开所有Walker可视化
多Walker2D模拟完成！


In [45]:
import jax
import jax.numpy as jnp
import numpy as np
import torch


def multi_walker_visualization(gnn_model, graph_builder, jit_inference_fn, n_walkers=2, n_steps=1000, device='cuda'):
    """
    为多个Walker2D生成可视化轨迹
    
    参数:
    - gnn_model: 预训练的GNN模型
    - graph_builder: WalkerGraphBuilder实例
    - jit_inference_fn: JAX JIT编译的RL推理函数
    - n_walkers: Walker数量
    - n_steps: 模拟步数
    - device: 计算设备
    
    返回:
    - 可视化HTML输出
    """
    # 创建多个Walker2D环境
    env_name = 'walker2d'
    backend = 'positional'
    walker_envs = []
    walker_resets = []
    walker_steps = []
    
    for _ in range(n_walkers):
        env = envs.create(env_name=env_name, backend=backend)
        walker_envs.append(env)
        walker_resets.append(jax.jit(env.reset))
        walker_steps.append(jax.jit(env.step))
    
    # 准备GNN模型
    gnn_model.to(device)
    gnn_model.eval()
    
    # 初始化随机种子
    base_rng = jax.random.PRNGKey(seed=1)
    rngs = []
    for i in range(n_walkers):
        base_rng, sub_rng = jax.random.split(base_rng)
        rngs.append(sub_rng)
    
    # 初始化环境状态
    states = []
    for i in range(n_walkers):
        states.append(walker_resets[i](rng=rngs[i]))
    
    # 数据收集
    all_data = [[] for _ in range(n_walkers)]
    all_rollouts = [[] for _ in range(n_walkers)]
    
    # 存储领导者历史动作 - 用于跟随者的延迟跟随
    leader_action_history = []
    
    # 辅助函数：进行GNN推理 - 保持与原始代码一致
    def gnn_inference(model, left_leg_data, graph_builder, device):
        """使用GNN模型预测右腿动作"""
        # 确保输入是PyTorch张量并增加批次维度
        if isinstance(left_leg_data, np.ndarray):
            left_leg_tensor = torch.tensor(left_leg_data, dtype=torch.float32)
        elif 'jax' in str(type(left_leg_data)):
            left_leg_tensor = torch.tensor(np.array(left_leg_data), dtype=torch.float32)
        else:
            left_leg_tensor = left_leg_data
            
        # 移至设备并确保有批次维度
        if len(left_leg_tensor.shape) == 1:
            left_leg_tensor = left_leg_tensor.unsqueeze(0)
        left_leg_tensor = left_leg_tensor.to(device)
        
        # 创建右腿占位符
        right_leg_dummy = torch.zeros_like(left_leg_tensor).to(device)
        
        # 构建图
        graph = graph_builder.build_graph_from_leg_data(left_leg_tensor.cpu(), right_leg_dummy.cpu())
        graph = graph.to(device)
        
        # 推理
        with torch.no_grad():
            output = model(graph)
            
        # 返回结果
        return output.cpu().numpy()
    
    # 模拟循环
    for step in range(n_steps):
        # 处理每个Walker
        for i in range(n_walkers):
            # 记录当前状态
            all_rollouts[i].append(states[i].pipeline_state)
            
            # 生成动作 - 按照与单Walker相同的方式
            if i == 0:  # 领导者使用RL策略
                rngs[i], act_rng = jax.random.split(rngs[i])
                act, _ = jit_inference_fn(states[i].obs, act_rng)
                
                # 重要：使用RL动作先更新一次状态
                states[i] = walker_steps[i](states[i], act)
                
                # 提取左腿动作
                left_leg_data = act[:3]
                
                # GNN预测右腿动作
                left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                
                # 创建完整动作
                action = np.concatenate([
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ])
                
                # 再次更新状态 - 使用完整动作
                states[i] = walker_steps[i](states[i], jnp.array(action))
                
                # 记录数据
                all_data[i].append((
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
                
                # 保存领导者动作供跟随者使用
                leader_action = np.array(act)
                leader_action_history.append(leader_action)
                
                # 保留最近20步的历史
                if len(leader_action_history) > 20:
                    leader_action_history.pop(0)
            
            else:  # 跟随者
                # 为跟随者生成更自然的跟随动作
                
                # 创建跟随动作 - 基于延迟跟随模式
                delay_steps = min(3 * i, len(leader_action_history) - 1)  # 延迟步数随距离增加
                
                if step < delay_steps:
                    # 启动阶段 - 使用较小振幅的动作
                    follower_action = np.array(leader_action) * 0.8
                else:
                    # 正常跟随 - 使用延迟的领导者动作
                    delayed_action = leader_action_history[-delay_steps-1]
                    
                    # 可以添加轻微的平滑处理
                    smooth_factor = 0.2
                    if step > delay_steps:
                        # 前一步的动作
                        prev_delayed_action = leader_action_history[-delay_steps]
                        # 平滑过渡
                        follower_action = (1 - smooth_factor) * np.array(delayed_action) + \
                                         smooth_factor * np.array(prev_delayed_action)
                    else:
                        follower_action = np.array(delayed_action)
                
                # 先使用基本动作更新状态
                states[i] = walker_steps[i](states[i], jnp.array(follower_action))
                
                # 提取左腿动作
                left_leg_data = follower_action[:3]
                
                # GNN预测右腿动作
                left_leg_tensor = torch.tensor(np.array(left_leg_data).copy(), dtype=torch.float32).unsqueeze(0).to(device)
                right_leg = gnn_inference(gnn_model, left_leg_tensor, graph_builder, device)
                
                # 创建完整动作
                action = np.concatenate([
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ])
                
                # 再次更新状态 - 使用完整动作
                states[i] = walker_steps[i](states[i], jnp.array(action))
                
                # 记录数据
                all_data[i].append((
                    left_leg_tensor.cpu().detach().numpy().squeeze(),
                    np.array(right_leg).squeeze()
                ))
                
                # 记录更新后的状态
                all_rollouts[i].append(states[i].pipeline_state)
            
            # 调试输出 - 每100步检查一次位置
            if step % 100 == 0:
                try:
                    position = np.array(states[i].pipeline_state.qp.pos[0])
                    print(f"Step {step}, Walker {i+1} position: {position}")
                except:
                    print(f"Step {step}, Walker {i+1} 无法获取位置")
    
    # 打印数据样本
    for i in range(n_walkers):
        print(f"Walker {i+1} 数据示例:", all_data[i][0])
    
    # 生成可视化 - 每个Walker一个单独的HTML文件
    for i in range(n_walkers):
        html_output = html.render(walker_envs[i].sys.tree_replace(
            {'opt.timestep': walker_envs[i].dt}), all_rollouts[i])
        
        with open(f"walker_{i+1}_output.html", "w") as f:
            f.write(html_output)
        
        print(f"Saved visualization for Walker {i+1} as walker_{i+1}_output.html")
    
    # 尝试创建组合可视化
    try:
        # 创建合并的轨迹数据
        combined_rollouts = []
        
        # 取每个步骤中所有Walker的状态
        for step in range(len(all_rollouts[0])):
            # 创建合并状态的列表
            step_states = []
            for i in range(n_walkers):
                # 调整位置，使Walker沿X轴排列
                adjusted_state = all_rollouts[i][step]
                step_states.append(adjusted_state)
            
            combined_rollouts.append(step_states)
        
        # 保存合并的轨迹数据
        with open("combined_trajectory_data.json", "w") as f:
            # 简单存储步数和Walker数量
            json.dump({"n_steps": len(combined_rollouts), "n_walkers": n_walkers}, f)
            
        print("保存了组合轨迹数据，可用于外部3D可视化")
    except Exception as e:
        print(f"创建组合可视化时出错: {e}")

    # 返回所有收集的数据
    return all_data, all_rollouts


# 实用函数：打开多个浏览器窗口显示
def open_multiple_visualizations(n_walkers):
    """打开多个浏览器窗口显示不同Walker的HTML"""
    import webbrowser
    import time
    
    for i in range(n_walkers):
        filename = f"walker_{i+1}_output.html"
        try:
            webbrowser.open(filename)
            time.sleep(0.5)  # 等待浏览器加载
        except Exception as e:
            print(f"无法打开{filename}：{e}")
    
    print("已打开所有Walker可视化")


# 主函数：运行多Walker模拟
if __name__ == "__main__":
    # 导入您的模型和图构建器
    # from your_model_file import SimpleWalkerGCN, WalkerGraphBuilder
    import torch
    import os
    import json
    
    # 加载GNN模型
    save_dir = "walker_gnn_results"
    checkpoint = torch.load(os.path.join(save_dir, "best_model.pth"))
    
    gnn_model = SimpleWalkerGCN(input_dim=1, hidden_dim=16, output_dim=3)
    gnn_model.load_state_dict(checkpoint['model_state_dict'])
    
    # 创建图构建器
    graph_builder = WalkerGraphBuilder()
    
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 获取RL推理函数
    # jit_inference_fn = jax.jit(inference_fn)  # 确保这个已定义
    
    # 运行多Walker模拟
    n_walkers = 2  # 设置Walker数量
    data, rollouts = multi_walker_visualization(
        gnn_model=gnn_model,
        graph_builder=graph_builder,
        jit_inference_fn=jit_inference_fn,
        n_walkers=n_walkers,
        device=device
    )
    
    # 打开所有Walker的可视化
    open_multiple_visualizations(n_walkers)
    
    print("多Walker2D模拟完成！")

Step 0, Walker 1 无法获取位置
Step 0, Walker 2 无法获取位置
Step 100, Walker 1 无法获取位置
Step 100, Walker 2 无法获取位置
Step 200, Walker 1 无法获取位置
Step 200, Walker 2 无法获取位置
Step 300, Walker 1 无法获取位置
Step 300, Walker 2 无法获取位置
Step 400, Walker 1 无法获取位置
Step 400, Walker 2 无法获取位置
Step 500, Walker 1 无法获取位置
Step 500, Walker 2 无法获取位置
Step 600, Walker 1 无法获取位置
Step 600, Walker 2 无法获取位置
Step 700, Walker 1 无法获取位置
Step 700, Walker 2 无法获取位置
Step 800, Walker 1 无法获取位置
Step 800, Walker 2 无法获取位置
Step 900, Walker 1 无法获取位置
Step 900, Walker 2 无法获取位置
Walker 1 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Walker 2 数据示例: (array([-0.969,  0.473,  0.928], dtype=float32), array([0.042, 0.144, 0.403], dtype=float32))
Saved visualization for Walker 1 as walker_1_output.html
Saved visualization for Walker 2 as walker_2_output.html
保存了组合轨迹数据，可用于外部3D可视化
已打开所有Walker可视化
多Walker2D模拟完成！


In [ ]:
jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gym
import jax
import time

# 兼容性修复：为新版NumPy创建bool8别名
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

# 模型定义
class EnhancedWalkerGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.2):
        super(EnhancedWalkerGNN, self).__init__()
        # 更深更宽的网络
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim*2)
        self.bn2 = nn.BatchNorm1d(hidden_dim*2)
        self.fc3 = nn.Linear(hidden_dim*2, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)
        
        # 添加残差连接
        self.shortcut = nn.Linear(input_dim, hidden_dim)
        
        # 权重初始化
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
    def forward(self, x):
        # 主路径
        main = F.leaky_relu(self.bn1(self.fc1(x)), negative_slope=0.1)
        main = self.dropout(main)
        main = F.leaky_relu(self.bn2(self.fc2(main)), negative_slope=0.1)
        main = self.dropout(main)
        main = F.leaky_relu(self.bn3(self.fc3(main)), negative_slope=0.1)
        
        # 残差连接
        shortcut = F.leaky_relu(self.shortcut(x), negative_slope=0.1)
        
        # 合并
        combined = main + shortcut
        output = self.fc4(combined)
        
        return output

def walker_to_ant_dimension(walker_action):
    """
    将Walker2D的3维动作转换为Ant的2维动作
    方法：线性投影，保留更多信息
    """
    # 定义投影矩阵 - 可以根据实际效果调整权重
    projection_matrix = np.array([
        [0.8, 0.2, 0.0],  # 髋关节映射 (主要来自Walker髋关节)
        [0.1, 0.7, 0.2]   # 膝关节映射 (主要来自Walker膝关节)
    ])
    
    # 执行投影
    return np.dot(projection_matrix, walker_action)

def symmetric_ant_control(first_leg_action, enhanced_model, device, phase_type=0):
    """
    使用EnhancedWalker模型通过对称映射控制Ant的四条腿
    
    参数:
    - first_leg_action: 第一条腿的Walker动作(3维)
    - enhanced_model: 训练好的EnhancedWalker模型
    - device: 计算设备
    - phase_type: 步态模式类型(0-3)
    
    返回:
    - 8维的Ant动作向量
    """
    # 步骤1: 使用模型获取对侧腿的动作
    enhanced_model.eval()
    with torch.no_grad():
        # 转为tensor并添加批次维度
        first_leg_action_np = np.array(first_leg_action)
        leg_tensor = torch.tensor(first_leg_action_np, dtype=torch.float32).unsqueeze(0).to(device)
        # 获取对侧腿动作
        # opposite_leg_tensor = enhanced_model(leg_tensor)
        opposite_leg_tensor = gnn_inference(gnn_model, leg_tensor, graph_builder)
        # 转回numpy并移除批次维度
        opposite_leg_action = np.array(opposite_leg_tensor).squeeze()
    
    # 步骤2: 将两条腿的Walker动作(3D)转换为Ant动作(2D)
    first_leg_ant = walker_to_ant_dimension(first_leg_action)
    opposite_leg_ant = walker_to_ant_dimension(opposite_leg_action)
    
    # 步骤3: 根据不同步态模式分配四条腿的动作
    ant_action = np.zeros(8)
    
    if phase_type == 0:  # 对角步态(前右+后左 vs 前左+后右)
        # 前右腿
        ant_action[0:2] = first_leg_ant
        # 前左腿
        ant_action[2:4] = opposite_leg_ant
        # 后右腿
        ant_action[4:6] = opposite_leg_ant  # 使用对侧腿动作
        # 后左腿
        ant_action[6:8] = first_leg_ant  # 使用原始腿动作
    
    elif phase_type == 1:  # 侧面步态(右侧 vs 左侧)
        # 前右腿
        ant_action[0:2] = first_leg_ant
        # 前左腿
        ant_action[2:4] = opposite_leg_ant
        # 后右腿
        ant_action[4:6] = first_leg_ant
        # 后左腿
        ant_action[6:8] = opposite_leg_ant
    
    elif phase_type == 2:  # 前后步态(前腿 vs 后腿)
        # 前右腿
        ant_action[0:2] = first_leg_ant
        # 前左腿
        ant_action[2:4] = first_leg_ant
        # 后右腿
        ant_action[4:6] = opposite_leg_ant
        # 后左腿
        ant_action[6:8] = opposite_leg_ant
    
    elif phase_type == 3:  # 爬行步态(更自然的四足动物步态)
        # 前右腿
        ant_action[0:2] = first_leg_ant
        # 前左腿
        ant_action[2:4] = -opposite_leg_ant  # 相反相位
        # 后右腿
        ant_action[4:6] = -first_leg_ant  # 相反相位
        # 后左腿
        ant_action[6:8] = opposite_leg_ant
    
    return ant_action

def run_ant_with_walker_model(gnn_model, num_steps=2000, render=True):
    """
    使用EnhancedWalker模型控制Ant环境的完整示例
    
    参数:
    - gnn_model: 训练好的模型
    - num_steps: 模拟步数
    - render: 是否渲染环境
    """
    # 设置设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    gnn_model.to(device)
    gnn_model.eval()

    
    
    # 创建Ant环境
    env = gym.make("Ant-v4")
    observation = env.reset()
    
    # 动作变化参数
    base_action = np.array([0.0, 0.0, 0.0])  # 初始静止状态
    action_scale = 0.5  # 动作幅度
    phase_type = 3  # 步态类型
    phase_change_freq = 200  # 每200步改变一次步态
    
    total_reward = 0
    for step in range(num_steps):
        # 每隔一定步数切换步态
        if step % phase_change_freq == 0:
            phase_type = (phase_type + 1) % 4
            print(f"切换到步态类型: {phase_type}")
        
        # 生成一个简单的周期性动作
        t = step / 20.0  # 时间参数
        # 简单的正弦波动作，模拟步行节奏
        first_leg_action = np.array([
            np.sin(t) * action_scale,
            np.cos(t) * action_scale * 0.5,
            np.sin(t * 2) * action_scale * 0.3
        ])
        
        # 生成Ant动作
        ant_action = symmetric_ant_control(first_leg_action, gnn_model, device, phase_type)
        
        # 执行动作 - 使用新版Gym API
        observation, reward, terminated, truncated, info = env.step(ant_action)
        total_reward += reward
        
        # 计算done标志
        done = terminated or truncated
        
        # 可视化
        if render:
            env.render()
        
        # 如果环境结束，重置
        if done:
            print(f"环境在第{step}步结束，总奖励: {total_reward}")
            observation = env.reset()
            total_reward = 0
    
    env.close()

# Ant与Walker2D集成的完整控制流程
def ant_walker_integration(walker_env, ant_env, enhanced_model, jit_inference_fn, num_steps=1000, render=True):
    """
    使用Walker2D环境的动作和增强模型共同控制Ant环境
    
    参数:
    - walker_env: 初始化的Walker2D环境
    - ant_env: 初始化的Ant环境
    - enhanced_model: 训练好的EnhancedWalker模型
    - jit_inference_fn: Walker环境的推理函数
    - num_steps: 模拟步数
    - render: 是否渲染环境
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    enhanced_model.to(device)
    enhanced_model.eval()
    
    # 初始化环境
    rng = jax.random.PRNGKey(seed=1)
    walker_state = jax.jit(walker_env.reset)(rng=rng)
    ant_observation = ant_env.reset()
    
    
    # 步态参数
    phase_type = 3
    phase_change_freq = 200
    
    total_reward = 0
    for step in range(num_steps):
        # 每隔一定步数切换步态
        if step % phase_change_freq == 0:
            phase_type = (phase_type + 1) % 4
            print(f"切换到步态类型: {phase_type}")
        
        # 从Walker环境获取动作
        act_rng, rng = jax.random.split(rng)
        walker_act, _ = jit_inference_fn(walker_state.obs, act_rng)
        
        # 使用Walker左腿动作
        left_leg_action = walker_act[:3]
        
        # 生成Ant动作
        ant_action = symmetric_ant_control(left_leg_action, enhanced_model, device, phase_type)
        
        # 在Ant环境中执行动作 - 使用新版Gym API
        ant_observation, reward, terminated, truncated, info = ant_env.step(ant_action)
        total_reward += reward
        
        # 计算done标志
        done = terminated or truncated
        
        # 更新Walker环境状态 - 可选，如果您需要保持Walker环境同步
        walker_state = jax.jit(walker_env.step)(walker_state, walker_act)
        
        # 可视化Ant环境
        if render:
            ant_env.render()
        
        # 如果Ant环境结束，重置
        if done:
            print(f"Ant环境在第{step}步结束，总奖励: {total_reward}")
            ant_observation = ant_env.reset()
            total_reward = 0
    
    ant_env.close()

# 使用示例
if __name__ == "__main__":
    # 假设您已经加载了训练好的模型
    # enhanced_model = torch.load('path_to_your_model.pth')
    
    # 方法1: 直接使用模型控制Ant
    # run_ant_with_walker_model(gnn_model, num_steps=2000)
    
    # 方法2: 使用Walker环境+增强模型控制Ant (需要导入相应模块)
    """
    # 初始化环境
    import jax
    from your_walker_module import envs
    
    walker_env = envs.create(env_name='walker2d', backend='positional')
    ant_env = gym.make("Ant-v4")
    
    # 集成控制
    ant_walker_integration(
        walker_env=walker_env,
        ant_env=ant_env,
        enhanced_model=enhanced_model,
        jit_inference_fn=jit_inference_fn,
        num_steps=2000
    )
    """

    import jax
    # from your_walker_module import envs
    
    walker_env = envs.create(env_name='walker2d', backend='positional')
    ant_env = gym.make("Ant-v4")
    
    # 集成控制
    ant_walker_integration(
        walker_env=walker_env,
        ant_env=ant_env,
        enhanced_model=gnn_model,
        jit_inference_fn=jit_inference_fn,
        num_steps=2000
    )